In [1]:
import json

with open('../data/task1.json', 'r') as f:
    test = json.load(f)


In [2]:
draft_ids = []
authors = []
coauthors = []
choices_2 = []
choices_3 = []
choices_4 = []
choices_5 = []

for instance in test:
    draft_ids.append(instance['folder_id'])
    authors.append(instance['author'])
    coauthors.append(instance['coauthor'])
    choices_2.append(instance['choices_2'])
    choices_3.append(instance['choices_3'])
    choices_4.append(instance['choices_4'])
    choices_5.append(instance['choices_5'])

In [3]:
import os

path = '../data/task1'
drafts = []
for i in draft_ids:
    folder_path = os.path.join(path, str(i))
    files = os.listdir(folder_path)
    json_file = [file for file in files if file.endswith('EN.json')][0]
    with open(os.path.join(folder_path, json_file)) as f:
        draft = json.load(f)
    drafts.append(draft['Content'])

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. Load environment variables from the .env file
load_dotenv()

# 2. Safely retrieve the API Key
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")

# Add a safety check to ensure the API key is loaded properly
if not SILICONFLOW_API_KEY:
    raise ValueError("API Key not found. Please ensure your .env file is created correctly and contains the SILICONFLOW_API_KEY variable!")

# 3. Configure the LLM using the SiliconFlow API
llm = ChatOpenAI(
    model="Pro/moonshotai/Kimi-K2.5", 
    api_key=SILICONFLOW_API_KEY,
    base_url="https://api.siliconflow.cn/v1", 
    temperature=0.0,
    max_tokens=20 # Restored from the original code to limit verbosity
)

/Users/liubowen/Desktop/CUHKcourse/2025-2026 Term2/Agentic AI for business and Fintech/individual project/UNBench-main/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
import random
from tqdm import tqdm
from langchain_core.messages import HumanMessage, SystemMessage

def run_task3_k_choices(llm, drafts, authors, choices_k):
    results = []
    invalid_responses = []
    
    # Iterate through the drafts, authors, and choices with a progress bar
    for i, (draft, author, choice_k) in tqdm(enumerate(zip(drafts, authors, choices_k)), total=len(drafts)):
        
        # Strictly restored the original system and user prompts
        system_prompt = f"""You are representing {author}, a country drafting a resolution for submission to the United Nations Security Council. 
        Your task is to review the draft resolution and select a coauthor from the following list: {', '.join(choice_k)}. 
        Respond only with the name of the chosen coauthor and provide no additional explanation."""
        
        user_prompt = f"""{author} is drafting a resolution to submit to the United Nations Security Council and is seeking a coauthor. 

        Review the following draft resolution and choose a coauthor from the list below. 
        Respond only with the name of the chosen country and provide no additional explanation.

        Available coauthors: {', '.join(choice_k)}

        Draft Resolution:
        {draft}

        Answer:
        """
        
        try:
            # Construct LangChain messages and invoke the model
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt)
            ]
            response = llm.invoke(messages)
            
            # Clean the generated text
            result = response.content.strip()
            
            # Strict validation: check if the result is exactly one of the valid choices
            valid_response = choice_k
            if result not in valid_response:
                # Fallback logic: check if any valid choice is mentioned in the model's output
                extracted = False
                for opt in valid_response:
                    if opt in result:
                        result = opt
                        extracted = True
                        break
                if not extracted:
                    print(f"Invalid response at index {i}: {result}")
                    result = random.choice(valid_response)
                    invalid_responses.append(i)
                    
        except Exception as e:
            # Catch API errors to prevent the loop from crashing and use a random choice
            print(f"Index {i} Error: {e}")
            result = random.choice(choice_k)
            invalid_responses.append(i)
            
        results.append(result)
        
    return results, invalid_responses

In [5]:
results_2, invalid_responses_2 = run_task3_k_choices(llm, drafts, authors, choices_2)

100%|██████████| 30/30 [16:25<00:00, 32.86s/it]


In [6]:
results_3, invalid_responses_3 = run_task3_k_choices(llm, drafts, authors, choices_3)

100%|██████████| 30/30 [22:28<00:00, 44.94s/it]


In [7]:
results_4, invalid_responses_4 = run_task3_k_choices(llm, drafts, authors, choices_4)

100%|██████████| 30/30 [26:00<00:00, 52.02s/it]


In [8]:
results_5, invalid_responses_5 = run_task3_k_choices(llm, drafts, authors, choices_5)

100%|██████████| 30/30 [26:29<00:00, 53.00s/it]


In [9]:
def calculate_accuracy(results, ground_truth):
    correct = 0
    for i in range(len(results)):
        if results[i] == ground_truth[i]:
            correct += 1
    return correct/len(results)

In [10]:
accuracy_2 = calculate_accuracy(results_2, coauthors)
accuracy_3 = calculate_accuracy(results_3, coauthors)
accuracy_4 = calculate_accuracy(results_4, coauthors)
accuracy_5 = calculate_accuracy(results_5, coauthors)

In [11]:
print(f'Accuracy for 2 choices: {accuracy_2}')
print(f'Accuracy for 3 choices: {accuracy_3}')
print(f'Accuracy for 4 choices: {accuracy_4}')
print(f'Accuracy for 5 choices: {accuracy_5}')

Accuracy for 2 choices: 0.6666666666666666
Accuracy for 3 choices: 0.6
Accuracy for 4 choices: 0.43333333333333335
Accuracy for 5 choices: 0.3333333333333333
